In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: gold_dim_customer
# ========================================

# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"

DOMAIN = "dimensions"
DATASET = "gold_dim_customer"

STORAGE_ACCOUNT = "stptfrozenfoodsdevwe01"
SILVER_CONTAINER = "silver"
GOLD_CONTAINER = "gold"

ORDERS_CUSTOMERS_DATASET = "silver_orders_customers"

ORDERS_CUSTOMERS_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{ORDERS_CUSTOMERS_DATASET}"

CANDIDATE_TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.dim_customer"

SILVER_BASE_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_BASE_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

ORDERS_CUSTOMERS_SOURCE_PATH = f"{SILVER_BASE_PATH}integration/{ORDERS_CUSTOMERS_DATASET}/"
CANDIDATE_TARGET_PATH = f"{GOLD_BASE_PATH}dimensions/dim_customer/"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING GOLD EXPLORATORY: gold_dim_customer")
print("=" * 80)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

STARTING GOLD EXPLORATORY: gold_dim_customer
[INFO] Context setup completed successfully.


In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("GOLD EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Target schema:                   {TARGET_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Orders customers table:          {ORDERS_CUSTOMERS_TABLE}")
print(f"Candidate target table:          {CANDIDATE_TARGET_TABLE}")
print(f"Orders customers source path:    {ORDERS_CUSTOMERS_SOURCE_PATH}")
print(f"Candidate target path:           {CANDIDATE_TARGET_PATH}")
print("=" * 80)

GOLD EXPLORATORY NOTEBOOK CONFIGURATION
Catalog:                         ptfrozenfoods_dev
Source schema:                   silver
Target schema:                   gold
Domain:                          dimensions
Dataset:                         gold_dim_customer
Orders customers table:          ptfrozenfoods_dev.silver.silver_orders_customers
Candidate target table:          ptfrozenfoods_dev.gold.dim_customer
Orders customers source path:    abfss://silver@stptfrozenfoodsdevwe01.dfs.core.windows.net/integration/silver_orders_customers/
Candidate target path:           abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/dimensions/dim_customer/


In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")
spark.sql(f"DESCRIBE TABLE {ORDERS_CUSTOMERS_TABLE}")

print("[INFO] Checking source path access...")
dbutils.fs.ls(ORDERS_CUSTOMERS_SOURCE_PATH)

print("[INFO] Checking target container access...")
dbutils.fs.ls(f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/")

print("[INFO] Pre-checks completed successfully.")

[INFO] Checking source table availability...
[INFO] Checking source path access...
[INFO] Checking target container access...
[INFO] Pre-checks completed successfully.


In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading source tables from the Silver layer...")

df_orders_customers = spark.table(ORDERS_CUSTOMERS_TABLE)

print("[INFO] Source tables loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Orders Customers:   {df_orders_customers.count():,}")
print("=" * 80)

# ========================================
# DATASET PREVIEW — INITIAL EXPLORATION
# ========================================

print("=" * 80)
print("DATASET PREVIEW — INITIAL EXPLORATION")
print("=" * 80)
print("[INFO] Displaying sample data from the source dataset...")
print("=" * 80)

print("\n[INFO] Preview: ORDERS AND CUSTOMERS (df_orders_customers)")
print("-" * 80)
display(df_orders_customers.limit(5))

print("\n" + "=" * 80)
print("[INFO] Dataset preview completed successfully.")
print("=" * 80)

[INFO] Loading source tables from the Silver layer...
[INFO] Source tables loaded successfully.
SOURCE DATA ROW COUNTS
Orders Customers:   89,600
DATASET PREVIEW — INITIAL EXPLORATION
[INFO] Displaying sample data from the source dataset...

[INFO] Preview: ORDERS AND CUSTOMERS (df_orders_customers)
--------------------------------------------------------------------------------


cliente_id,pedido_id,data_pedido,canal_id,vendedor_id,cidade_entrega,estado_pedido,prazo_entrega_dias,observacao_pedido,sistema_origem,usuario_ultima_alteracao,order_load_date,order_ingestion_timestamp,order_source_file,nome_cliente,tipo_cliente,nif,data_registo,cliente_cidade,distrito,canal_captacao,score_atividade,obs_comercial,segmento,potencial_valor,cluster_comercial,status_cliente,data_status,motivo_status
C0136,PED2021011100001,2021-01-11,CH03,VEND008,Espinho,Entregue,2,null,ERP_FROZEN_V2,maria.s,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,Restaurante 136,Restaurante,243297765,2021-01-11,Espinho,Aveiro,Telefone,0.996,null,Pequeno,Alto,Cluster C,Ativo,2026-02-28,null
C0136,PED2021011100002,2021-01-11,CH02,VEND001,Espinho,Entregue,2,null,ERP_FROZEN_V2,maria.s,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,Restaurante 136,Restaurante,243297765,2021-01-11,Espinho,Aveiro,Telefone,0.996,null,Pequeno,Alto,Cluster C,Ativo,2026-02-28,null
C0136,PED2021011100003,2021-01-11,CH03,null,Espinho,Entregue,2,null,ERP_FROZEN_V2,ana.c,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,Restaurante 136,Restaurante,243297765,2021-01-11,Espinho,Aveiro,Telefone,0.996,null,Pequeno,Alto,Cluster C,Ativo,2026-02-28,null
C0136,PED2021011100004,2021-01-11,CH01,VEND008,Espinho,Entregue,2,null,ERP_FROZEN_V2,svc_sync,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,Restaurante 136,Restaurante,243297765,2021-01-11,Espinho,Aveiro,Telefone,0.996,null,Pequeno,Alto,Cluster C,Ativo,2026-02-28,null
C0136,PED2021011100007,2021-01-11,CH03,VEND001,Espinho,Entregue,3,null,ERP_FROZEN_V2,svc_sync,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,Restaurante 136,Restaurante,243297765,2021-01-11,Espinho,Aveiro,Telefone,0.996,null,Pequeno,Alto,Cluster C,Ativo,2026-02-28,null



[INFO] Dataset preview completed successfully.


In [0]:
# ========================================
# 5. SOURCE VALIDATION
# ========================================

from pyspark.sql import functions as F

print("[INFO] Printing source schema...")
df_orders_customers.printSchema()

required_columns = [
    "cliente_id",
    "tipo_cliente",
    "segmento",
    "cliente_cidade",
    "distrito",
    "status_cliente"
]

missing_columns = [c for c in required_columns if c not in df_orders_customers.columns]

print(f"[INFO] Missing required columns: {missing_columns}")

if missing_columns:
    raise Exception(f"Missing required columns: {missing_columns}")

print("[INFO] Source validation completed successfully.")

[INFO] Printing source schema...
root
 |-- cliente_id: string (nullable = true)
 |-- pedido_id: string (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- canal_id: string (nullable = true)
 |-- vendedor_id: string (nullable = true)
 |-- cidade_entrega: string (nullable = true)
 |-- estado_pedido: string (nullable = true)
 |-- prazo_entrega_dias: integer (nullable = true)
 |-- observacao_pedido: string (nullable = true)
 |-- sistema_origem: string (nullable = true)
 |-- usuario_ultima_alteracao: string (nullable = true)
 |-- order_load_date: date (nullable = true)
 |-- order_ingestion_timestamp: timestamp (nullable = true)
 |-- order_source_file: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- nif: string (nullable = true)
 |-- data_registo: date (nullable = true)
 |-- cliente_cidade: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- canal_captacao: string (nullable = true)
 |-- score_

In [0]:
# ========================================
# 6. INITIAL DATA QUALITY CHECKS
# ========================================

display(
    df_orders_customers.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("cliente_id").alias("distinct_cliente_id"),
        F.sum(F.when(F.col("cliente_id").isNull(), 1).otherwise(0)).alias("null_cliente_id")
    )
)

total_rows,distinct_cliente_id,null_cliente_id
89600,145,0


In [0]:
# ========================================
# 7. DUPLICATE CHECK — CUSTOMERS
# ========================================

df_duplicates = (
    df_orders_customers
    .groupBy("cliente_id")
    .count()
    .filter(F.col("count") > 1)
)

display(df_duplicates)

cliente_id,count
C0059,356
C0195,953
C0167,35
C0150,312
C0016,253
C0067,162
C0131,10
C0191,84
C0057,3880
C0280,19


In [0]:
# ========================================
# 8. BUSINESS PROFILE CHECKS
# ========================================

print("[INFO] Customer type distribution")
display(df_orders_customers.groupBy("tipo_cliente").count())

print("[INFO] Customer segment distribution")
display(df_orders_customers.groupBy("segmento").count())

print("[INFO] Customer status distribution")
display(df_orders_customers.groupBy("status_cliente").count())

print("[INFO] Customer geographic distribution")
display(df_orders_customers.groupBy("distrito").count())

[INFO] Customer type distribution


tipo_cliente,count
Take-away,6196
Supermercado,27689
Hotel,15595
Restaurante,39004
Particular,1116


[INFO] Customer segment distribution


segmento,count
Pequeno,15692
Médio,33684
Grande,40224


[INFO] Customer status distribution


status_cliente,count
Inativo,1037
Ativo,84345
Dormante,4218


[INFO] Customer geographic distribution


distrito,count
Braga,21171
Aveiro,12136
Viana do Castelo,6836
Porto,49457


In [0]:
# ========================================
# 9. BUILD CANDIDATE DIMENSION
# ========================================

df_dim_customer_candidate = (
    df_orders_customers
    .select(
        "cliente_id",
        "tipo_cliente",
        "segmento",
        "cliente_cidade",
        "distrito",
        "potencial_valor",
        "cluster_comercial",
        "status_cliente"
    )
    .dropDuplicates(["cliente_id"])
)

print(f"[INFO] Candidate dimension row count: {df_dim_customer_candidate.count():,}")

display(df_dim_customer_candidate.limit(10))

[INFO] Candidate dimension row count: 145


cliente_id,tipo_cliente,segmento,cliente_cidade,distrito,potencial_valor,cluster_comercial,status_cliente
C0059,Restaurante,Pequeno,Valongo,Porto,Baixo,Cluster C,Ativo
C0195,Hotel,Grande,Porto,Porto,Médio,Cluster B,Ativo
C0167,Particular,Pequeno,Vila Nova de Gaia,Porto,Baixo,Cluster B,Ativo
C0150,Take-away,Médio,Barcelos,Braga,Médio,Cluster B,Ativo
C0016,Supermercado,Pequeno,Valongo,Porto,Médio,Cluster C,Ativo
C0067,Supermercado,Pequeno,Santo Tirso,Porto,Alto,Cluster A,Dormante
C0131,Restaurante,Pequeno,Viana do Castelo,Viana do Castelo,Médio,Cluster A,Inativo
C0191,Take-away,Pequeno,Valongo,Porto,Alto,Cluster C,Ativo
C0057,Restaurante,Médio,Santo Tirso,Porto,Baixo,Cluster C,Ativo
C0280,Restaurante,Pequeno,Vila Nova de Gaia,Porto,Médio,Cluster C,Inativo


In [0]:
# ========================================
# 10. CANDIDATE DIMENSION VALIDATION
# ========================================

display(
    df_dim_customer_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("cliente_id").alias("distinct_cliente_id"),
        F.sum(F.when(F.col("cliente_id").isNull(), 1).otherwise(0)).alias("null_cliente_id")
    )
)

print("[INFO] Candidate dimension validation completed successfully.")

total_rows,distinct_cliente_id,null_cliente_id
145,145,0


[INFO] Candidate dimension validation completed successfully.


In [0]:
# ========================================
# 13. MISSING VALUES ANALYSIS — GENERIC
# ========================================

from pyspark.sql import functions as F

def analyze_missing_values(df, dataset_name):
    print("=" * 80)
    print(f"MISSING VALUES ANALYSIS — {dataset_name.upper()}")
    print("=" * 80)

    total_rows = df.count()

    missing_values_df = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

    missing_values_transposed = (
        missing_values_df
        .select(
            F.array([
                F.struct(
                    F.lit(col_name).alias("column_name"),
                    F.col(col_name).alias("null_count")
                ) for col_name in missing_values_df.columns
            ]).alias("missing_data")
        )
        .select(F.explode("missing_data").alias("data"))
        .select(
            F.col("data.column_name"),
            F.col("data.null_count")
        )
        .withColumn(
            "null_percentage",
            F.round((F.col("null_count") / F.lit(total_rows)) * 100, 4)
        )
        .orderBy(F.col("null_percentage").desc())
    )

    display(missing_values_transposed)

    print(f"[INFO] Total rows analyzed: {total_rows:,}")
    print("[INFO] Missing values analysis completed successfully.")
    print("=" * 80)

analyze_missing_values(df_dim_customer_candidate, "dim_customer")

MISSING VALUES ANALYSIS — DIM_CUSTOMER


column_name,null_count,null_percentage
cliente_id,0,0.0
distrito,0,0.0
segmento,0,0.0
cliente_cidade,0,0.0
potencial_valor,0,0.0
tipo_cliente,0,0.0
cluster_comercial,0,0.0
status_cliente,0,0.0


[INFO] Total rows analyzed: 145
[INFO] Missing values analysis completed successfully.
